In [1]:
## Cell 1 — Instalasi pustaka

!pip -q install ucimlrepo plotly ipywidgets openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 41.2 MB/s eta 0:00:00


In [2]:
## Cell 2 — Import pustaka

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output
from ucimlrepo import fetch_ucirepo

pd.set_option("display.max_columns", None)

print("Semua pustaka berhasil diimpor.")

Semua pustaka berhasil diimpor.


In [3]:
## Cell 3 — Mengunduh dataset publik

# Mengambil dataset Online Retail dari UCI
retail = fetch_ucirepo(id=352)

# Mengambil data asli beserta kolom-kolomnya
df = retail.data.original.copy()

print("Ukuran dataset:", df.shape)
print("\nNama kolom:")
print(df.columns.tolist())

display(df.head())

Ukuran dataset: (541909, 8)

Nama kolom:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [4]:
## Cell 4 — Memeriksa kualitas data

print("Informasi dataset:")
df.info()

print("\nJumlah nilai kosong setiap kolom:")
display(df.isnull().sum().to_frame("Jumlah Missing Value"))

print("\nJumlah baris duplikat:", df.duplicated().sum())

print("\nStatistik kuantitas dan harga:")
display(df[["Quantity", "UnitPrice"]].describe())


Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB

Jumlah nilai kosong setiap kolom:


,Jumlah Missing Value
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0



Jumlah baris duplikat: 5268

Statistik kuantitas dan harga:


,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


In [5]:
## Cell 5 — Membersihkan dan menyiapkan data

# Salinan data agar data asli tidak berubah
sales = df.copy()

# Memastikan format kolom sesuai
sales["InvoiceNo"] = sales["InvoiceNo"].astype("string").str.strip()
sales["Description"] = sales["Description"].astype("string").str.strip()
sales["Country"] = sales["Country"].astype("string").str.strip()

sales["InvoiceDate"] = pd.to_datetime(
    sales["InvoiceDate"], errors="coerce"
)

sales["Quantity"] = pd.to_numeric(
    sales["Quantity"], errors="coerce"
)

sales["UnitPrice"] = pd.to_numeric(
    sales["UnitPrice"], errors="coerce"
)

# Menghapus baris tanpa informasi penting
sales = sales.dropna(
    subset=[
        "InvoiceNo", "StockCode", "Description",
        "InvoiceDate", "Quantity", "UnitPrice", "Country"
    ]
)

# Mengecualikan faktur pembatalan
sales = sales[
    ~sales["InvoiceNo"].str.upper().str.startswith("C", na=False)
]

# Mempertahankan transaksi dengan kuantitas dan harga positif
sales = sales[
    (sales["Quantity"] > 0) &
    (sales["UnitPrice"] > 0)
].copy()

# Menghitung nilai penjualan setiap baris transaksi
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]

# Membuat kolom waktu untuk analisis
sales["YearMonth"] = sales["InvoiceDate"].dt.to_period("M").astype(str)
sales["DayName"] = sales["InvoiceDate"].dt.day_name()
sales["Date"] = sales["InvoiceDate"].dt.date

# Mengubah urutan hari agar grafik mudah dibaca
day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

print("Jumlah baris setelah pembersihan:", len(sales))
print("Rentang tanggal:", sales["InvoiceDate"].min(),
      "sampai", sales["InvoiceDate"].max())
print("Jumlah negara:", sales["Country"].nunique())
print("Jumlah produk:", sales["StockCode"].nunique())

display(sales.head())


Jumlah baris setelah pembersihan: 530104
Rentang tanggal: 2010-12-01 08:26:00 sampai 2011-12-09 12:50:00
Jumlah negara: 38
Jumlah produk: 3922


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,YearMonth,DayName,Date
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010-12,Wednesday,2010-12-01
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,2010-12-01
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010-12,Wednesday,2010-12-01
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,2010-12-01
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010-12,Wednesday,2010-12-01


In [6]:
## Cell 6 — Membuat ringkasan agregat

# Ringkasan penjualan bulanan
monthly_sales = (
    sales.groupby("YearMonth", as_index=False)
    .agg(
        Revenue=("Revenue", "sum"),
        Total_Units=("Quantity", "sum"),
        Transactions=("InvoiceNo", "nunique")
    )
    .sort_values("YearMonth")
)

# Ringkasan produk
product_sales = (
    sales.groupby("Description", as_index=False)
    .agg(
        Revenue=("Revenue", "sum"),
        Total_Units=("Quantity", "sum"),
        Transactions=("InvoiceNo", "nunique")
    )
    .sort_values("Revenue", ascending=False)
)

# Ringkasan negara
country_sales = (
    sales.groupby("Country", as_index=False)
    .agg(
        Revenue=("Revenue", "sum"),
        Transactions=("InvoiceNo", "nunique")
    )
    .sort_values("Revenue", ascending=False)
)

print("Ringkasan bulanan:")
display(monthly_sales.head())

print("Lima produk dengan penjualan terbesar:")
display(product_sales.head())

print("Lima negara dengan nilai penjualan terbesar:")
display(country_sales.head())


Ringkasan bulanan:


,YearMonth,Revenue,Total_Units,Transactions
0,2010-12,823746.140,359239,1559
1,2011-01,691364.560,387785,1086
2,2011-02,523631.890,283555,1100
3,2011-03,717639.360,377526,1454
4,2011-04,537808.621,308815,1246


Lima produk dengan penjualan terbesar:


,Description,Revenue,Total_Units,Transactions
1056,DOTCOM POSTAGE,206248.77,706,706
2842,REGENCY CAKESTAND 3 TIER,174484.74,13879,1988
2378,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1
3833,WHITE HANGING HEART T-LIGHT HOLDER,106292.77,37891,2256
2403,PARTY BUNTING,99504.33,18295,1685


Lima negara dengan nilai penjualan terbesar:


,Country,Revenue,Transactions
36,United Kingdom,9025222.084,18019
24,Netherlands,285446.340,94
10,EIRE,283453.960,288
14,Germany,228867.140,457
13,France,209715.110,392


In [7]:
## Cell 7 — Visualisasi tren penjualan bulanan

fig = px.line(
    monthly_sales,
    x="YearMonth",
    y="Revenue",
    markers=True,
    title="Tren Penjualan Bulanan",
    labels={
        "YearMonth": "Bulan",
        "Revenue": "Nilai Penjualan (£)"
    },
    template="plotly_white"
)

fig.update_layout(
    xaxis_title="Bulan",
    yaxis_title="Nilai Penjualan (£)",
    hovermode="x unified"
)

fig.show()


In [8]:
## Cell 8 — Visualisasi produk terlaris


top_products = product_sales.head(10).sort_values("Revenue")

fig = px.bar(
    top_products,
    x="Revenue",
    y="Description",
    orientation="h",
    title="10 Produk dengan Nilai Penjualan Terbesar",
    labels={
        "Revenue": "Nilai Penjualan (£)",
        "Description": "Nama Produk"
    },
    template="plotly_white"
)

fig.update_layout(yaxis_title="Produk")
fig.show()


In [9]:
## Cell 9 — Visualisasi pola penjualan berdasarkan hari


daily_sales = (
    sales.groupby("DayName", as_index=False)
    .agg(
        Revenue=("Revenue", "sum"),
        Transactions=("InvoiceNo", "nunique")
    )
)

daily_sales["DayName"] = pd.Categorical(
    daily_sales["DayName"],
    categories=day_order,
    ordered=True
)

daily_sales = daily_sales.sort_values("DayName")

fig = px.bar(
    daily_sales,
    x="DayName",
    y="Revenue",
    title="Nilai Penjualan Berdasarkan Hari",
    labels={
        "DayName": "Hari",
        "Revenue": "Nilai Penjualan (£)"
    },
    template="plotly_white"
)

fig.update_layout(xaxis_title="Hari")
fig.show()


In [10]:
## Cell 10 — Membuat kontrol dasbor interaktif


country_options = ["Semua Negara"] + sorted(
    sales["Country"].dropna().unique().tolist()
)

country_filter = widgets.Dropdown(
    options=country_options,
    value="Semua Negara",
    description="Negara:",
    layout=widgets.Layout(width="450px"),
    style={"description_width": "80px"}
)

top_n_filter = widgets.IntSlider(
    value=10,
    min=5,
    max=20,
    step=5,
    description="Top Produk:",
    continuous_update=False,
    style={"description_width": "100px"},
    layout=widgets.Layout(width="450px")
)

print("Kontrol dasbor berhasil dibuat.")


Kontrol dasbor berhasil dibuat.


In [11]:
## Cell 11 — Dasbor interaktif penjualan
# Prasyarat: Cell 2 (impor), Cell 5 (DataFrame sales), Cell 10 (country_filter, top_n_filter)

import base64
import io
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

WARNA_UTAMA = "#244c3b"

# ---------- 1. Fungsi perhitungan (tanpa tampilan) ----------
def hitung_dashboard(country, top_n):
    """Menghitung KPI, tren bulanan, dan produk terlaris untuk filter yang dipilih."""
    data = sales if country == "Semua Negara" else sales[sales["Country"] == country]
    if data.empty:
        return None

    kpi = {
        "revenue": float(data["Revenue"].sum()),
        "invoices": int(data["InvoiceNo"].nunique()),
        "units": int(data["Quantity"].sum()),
        "products": int(data["Description"].nunique()),
    }

    monthly = (
        data.groupby("YearMonth", as_index=False)["Revenue"].sum()
        .sort_values("YearMonth")
    )

    top_products = (
        data.groupby("Description", as_index=False)
        .agg(Revenue=("Revenue", "sum"), Quantity=("Quantity", "sum"))
        .sort_values("Revenue", ascending=False)
        .head(int(top_n))
        .reset_index(drop=True)
    )
    return kpi, monthly, top_products


# ---------- 2. Fungsi tampilan ----------
def html_dashboard(country, kpi):
    def kartu(label, nilai, warna):
        return f"""
        <div style="background:{warna};padding:16px 18px;border-radius:12px;
                    border:1px solid rgba(0,0,0,.06);">
          <div style="font-size:13px;color:#4b5563;">{label}</div>
          <div style="font-size:24px;font-weight:700;color:{WARNA_UTAMA};
                      margin-top:4px;">{nilai}</div>
        </div>"""

    return f"""
    <div style="font-family:Arial,sans-serif;margin:8px 0 4px 0;">
      <div style="font-size:24px;font-weight:700;color:{WARNA_UTAMA};">
        Sales Performance Dashboard</div>
      <div style="color:#6b7280;margin:2px 0 14px 0;">
        Analisis transaksi penjualan &middot; Wilayah: <b>{country}</b></div>
      <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));
                  gap:12px;">
        {kartu("Total Penjualan", f"£{kpi['revenue']:,.2f}", "#e8f3ec")}
        {kartu("Total Transaksi", f"{kpi['invoices']:,}", "#eaf1fa")}
        {kartu("Jumlah Unit Terjual", f"{kpi['units']:,}", "#fff2d9")}
        {kartu("Jenis Produk", f"{kpi['products']:,}", "#f1eafa")}
      </div>
    </div>"""


def buat_grafik(monthly, top_products, top_n):
    fig1 = px.line(
        monthly, x="YearMonth", y="Revenue", markers=True,
        title="Tren Penjualan Bulanan", template="plotly_white",
        labels={"YearMonth": "Bulan", "Revenue": "Penjualan (£)"},
    )
    fig1.update_traces(line=dict(width=3, color=WARNA_UTAMA), marker=dict(size=8))
    fig1.update_yaxes(tickprefix="£", tickformat=",.0f")
    fig1.update_layout(height=400, margin=dict(l=25, r=25, t=70, b=45),
                       hovermode="x unified")

    fig2 = px.bar(
        top_products.sort_values("Revenue"), x="Revenue", y="Description",
        orientation="h", template="plotly_white",
        title=f"{int(top_n)} Produk dengan Penjualan Tertinggi",
        labels={"Revenue": "Penjualan (£)", "Description": "Nama Produk"},
    )
    fig2.update_traces(marker_color=WARNA_UTAMA)
    fig2.update_xaxes(tickprefix="£", tickformat=",.0f")
    fig2.update_yaxes(automargin=True, title=None)
    fig2.update_layout(height=max(420, int(top_n) * 28),
                       margin=dict(l=25, r=25, t=70, b=45))
    return fig1, fig2


def buat_tabel(top_products):
    tabel = top_products.rename(columns={
        "Description": "Nama Produk",
        "Quantity": "Jumlah Terjual",
        "Revenue": "Penjualan (£)",
    })
    tabel.insert(0, "No", range(1, len(tabel) + 1))
    return (
        tabel.style
        .format({"Jumlah Terjual": "{:,.0f}", "Penjualan (£)": "£{:,.2f}"})
        .set_table_styles([
            {"selector": "th",
             "props": [("background-color", WARNA_UTAMA), ("color", "white"),
                       ("text-align", "left"), ("padding", "8px 12px"),
                       ("font-family", "Arial, sans-serif")]},
            {"selector": "td", "props": [("padding", "6px 12px")]},
        ])
        .set_properties(subset=["Nama Produk"], **{"text-align": "left"})
        .hide(axis="index")
    )


def _png_matplotlib(fig):
    import matplotlib.pyplot as plt
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=130)
    plt.close(fig)
    return buf.getvalue()


def grafik_matplotlib(monthly, top_products):
    """Cadangan bila Plotly tidak bisa dijadikan gambar (kaleido tidak tersedia)."""
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter
    rupiah = FuncFormatter(lambda v, _: f"£{v:,.0f}")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(monthly["YearMonth"], monthly["Revenue"], marker="o", lw=3, color=WARNA_UTAMA)
    ax.set_title("Tren Penjualan Bulanan", loc="left", fontsize=13, fontweight="bold")
    ax.set_xlabel("Bulan"); ax.set_ylabel("Penjualan (£)")
    ax.yaxis.set_major_formatter(rupiah); ax.grid(alpha=.3); ax.set_ylim(bottom=0)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    fig.tight_layout()
    png1 = _png_matplotlib(fig)

    data = top_products.sort_values("Revenue")
    fig, ax = plt.subplots(figsize=(10, max(4, len(data) * 0.4)))
    ax.barh(data["Description"].astype(str), data["Revenue"], color=WARNA_UTAMA)
    ax.set_title(f"{len(data)} Produk dengan Penjualan Tertinggi", loc="left",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Penjualan (£)"); ax.xaxis.set_major_formatter(rupiah)
    ax.grid(axis="x", alpha=.3)
    fig.tight_layout()
    png2 = _png_matplotlib(fig)
    return [png1, png2]


def buat_gambar_grafik(monthly, top_products, top_n):
    """Plotly -> gambar PNG (jika kaleido ada); jika gagal pakai Matplotlib."""
    if PLOTLY_PNG:
        try:
            fig1, fig2 = buat_grafik(monthly, top_products, top_n)
            return [fig.to_image(format="png", width=1000,
                                 height=int(fig.layout.height), scale=2)
                    for fig in (fig1, fig2)]
        except Exception:
            pass
    return grafik_matplotlib(monthly, top_products)


def html_gambar(daftar_png):
    return "".join(
        '<img src="data:image/png;base64,' + base64.b64encode(p).decode() + '" '
        'style="width:100%;max-width:1000px;display:block;margin:12px 0;">'
        for p in daftar_png
    )


def _uji_plotly_png():
    try:
        px.line(x=[1, 2], y=[1, 2]).to_image(format="png", width=100, height=100)
        return True
    except Exception:
        return False


PLOTLY_PNG = _uji_plotly_png()
print("Mesin grafik dasbor:",
      "Plotly (gambar PNG)" if PLOTLY_PNG else "Matplotlib (cadangan; kaleido tidak tersedia)")


# ---------- 3. Komponen dasbor ----------
kpi_box = widgets.HTML()
grafik_box = widgets.HTML()
dashboard_output = widgets.Output()


def update_dashboard(country, top_n):
    hasil = hitung_dashboard(country, top_n)      # dihitung di luar Output agar error terlihat

    if hasil is None:
        kpi_box.value = "<h3>Data tidak ditemukan untuk pilihan ini.</h3>"
        grafik_box.value = ""
        dashboard_output.clear_output()
        return

    kpi, monthly, top_products = hasil
    kpi_box.value = html_dashboard(country, kpi)
    grafik_box.value = html_gambar(buat_gambar_grafik(monthly, top_products, top_n))

    dashboard_output.clear_output(wait=True)
    with dashboard_output:
        display(HTML(f"<h3 style='color:{WARNA_UTAMA};font-family:Arial,sans-serif;'>"
                     "Ringkasan Produk Terlaris</h3>"))
        display(buat_tabel(top_products))


# ---------- 4. Hubungkan filter (aman dijalankan ulang) ----------
if "refresh_dashboard" in globals():                # lepas handler lama bila cell dijalankan ulang
    for w in (country_filter, top_n_filter):
        try:
            w.unobserve(refresh_dashboard, names="value")
        except ValueError:
            pass


def refresh_dashboard(change=None):
    update_dashboard(country_filter.value, top_n_filter.value)


country_filter.observe(refresh_dashboard, names="value")
top_n_filter.observe(refresh_dashboard, names="value")

# ---------- 5. Tampilkan ----------
display(widgets.VBox([
    widgets.VBox([country_filter, top_n_filter]),
    kpi_box,
    grafik_box,
    dashboard_output,
]))
refresh_dashboard()


Mesin grafik dasbor: Matplotlib (cadangan; kaleido tidak tersedia)


In [12]:
## Cell 12 — Pengujian hasil pengolahan data
# Memastikan angka dasbor konsisten dengan data hasil pembersihan.

hasil_uji = []

# 1. Total penjualan: ringkasan bulanan harus sama dengan data utama
hasil_uji.append((
    "Total penjualan bulanan = total data sales",
    abs(monthly_sales["Revenue"].sum() - sales["Revenue"].sum()) < 0.01
))

# 2. KPI 'Semua Negara' harus sama dengan perhitungan langsung
kpi_all, monthly_all, top_all = hitung_dashboard("Semua Negara", 10)
hasil_uji.append(("KPI penjualan 'Semua Negara' cocok",
                  abs(kpi_all["revenue"] - sales["Revenue"].sum()) < 0.01))
hasil_uji.append(("KPI transaksi 'Semua Negara' cocok",
                  kpi_all["invoices"] == sales["InvoiceNo"].nunique()))

# 3. Jumlah produk teratas mengikuti slider
for n in (5, 10, 15, 20):
    _, _, top_n_df = hitung_dashboard("Semua Negara", n)
    hasil_uji.append((f"Top produk = {n} baris", len(top_n_df) == n))

# 4. Filter negara benar-benar memfilter data
negara_uji = "United Kingdom" if "United Kingdom" in country_options else country_options[1]
kpi_negara, _, _ = hitung_dashboard(negara_uji, 10)
hasil_uji.append((f"Filter negara '{negara_uji}' berfungsi",
                  kpi_negara["revenue"] <= kpi_all["revenue"]))

for nama, lolos in hasil_uji:
    print(("LOLOS  " if lolos else "GAGAL  ") + nama)

print("\nSemua pengujian lolos." if all(l for _, l in hasil_uji)
      else "\nAda pengujian yang gagal, periksa baris bertanda GAGAL.")


LOLOS  Total penjualan bulanan = total data sales
LOLOS  KPI penjualan 'Semua Negara' cocok
LOLOS  KPI transaksi 'Semua Negara' cocok
LOLOS  Top produk = 5 baris
LOLOS  Top produk = 10 baris
LOLOS  Top produk = 15 baris
LOLOS  Top produk = 20 baris
LOLOS  Filter negara 'United Kingdom' berfungsi

Semua pengujian lolos.


In [13]:
## Cell 13 — Menyimpan ringkasan hasil ke CSV dan dasbor ke HTML
# File HTML dapat dibuka di browser tanpa Colab (berguna untuk GitHub / lampiran tugas).

monthly_sales.to_csv("ringkasan_penjualan_bulanan.csv", index=False)
product_sales.to_csv("ringkasan_penjualan_produk.csv", index=False)
country_sales.to_csv("ringkasan_penjualan_negara.csv", index=False)


def ekspor_dashboard_html(country="Semua Negara", top_n=10,
                          nama_file="dashboard_penjualan.html"):
    kpi, monthly, top_products = hitung_dashboard(country, top_n)
    fig1, fig2 = buat_grafik(monthly, top_products, top_n)

    halaman = f"""<!DOCTYPE html>
<html lang="id"><head><meta charset="utf-8">
<title>Sales Performance Dashboard</title></head>
<body style="max-width:1100px;margin:auto;padding:16px;font-family:Arial,sans-serif;">
{html_dashboard(country, kpi)}
{fig1.to_html(full_html=False, include_plotlyjs="cdn")}
{fig2.to_html(full_html=False, include_plotlyjs=False)}
<h3 style="color:{WARNA_UTAMA};">Ringkasan Produk Terlaris</h3>
{buat_tabel(top_products).to_html()}
</body></html>"""

    with open(nama_file, "w", encoding="utf-8") as f:
        f.write(halaman)
    return nama_file


print("File tersimpan:", ekspor_dashboard_html())
# Untuk mengunduh dari Colab, hapus tanda # pada dua baris berikut:
# from google.colab import files
# files.download("dashboard_penjualan.html")


File tersimpan: dashboard_penjualan.html
